<a href="https://colab.research.google.com/github/mmuputisi/Adv-Py_Data_Analysis/blob/main/Visual%20-%20MA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Create the file using Python's file writing with triple-single quotes
file_content = '''"""
Annex C Visualizations and Tables for Moving Average Models
INTEGRATED VERSION - Uses existing data from main analyzer
"""
'''
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# CONFIGURATION
# ============================================================================
class AnnexConfig:
    """Configuration for Annex visualizations"""
    OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/FTSE Data/annex_figures'

    # Color scheme
    COLORS = {
        'cs': '#2E86AB',      # Blue for Consumer Staples
        'all': '#A23B72',      # Purple for All Sectors
        'E': '#2E86AB',        # Blue
        'S': '#A23B72',        # Purple
        'G': '#F18F01',         # Orange
        'line': '#000000',      # Black for lines
        'hist': '#4C9A8A'       # Teal for histograms
    }


# ============================================================================
# FIGURE 1: ESG TRENDS OVER TIME
# ============================================================================
def create_esg_trend_figures(df_cs, df_all, config):
    """Create ESG trend visualizations using existing dataframes"""

    print("\n" + "="*60)
    print("CREATING ESG TREND VISUALIZATIONS")
    print("="*60)

    # Reset index to get years as column
    df_cs_reset = df_cs.reset_index()
    df_all_reset = df_all.reset_index()

    # Check which ESG columns exist
    available_components = [c for c in ['E', 'S', 'G'] if c in df_cs.columns]
    print(f"Available ESG components: {available_components}")

    if not available_components:
        print("⚠ No ESG components found - skipping trend figures")
        return None, None, None

    # Calculate yearly averages by sector
    yearly_cs = df_cs_reset.groupby('Year')[available_components].mean().reset_index()
    yearly_all = df_all_reset.groupby('Year')[available_components].mean().reset_index()

    # FIGURE 1: Overall ESG Trend (Consumer Staples vs All Sectors)
    fig1, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig1.suptitle('ESG Trends Over Time', fontsize=16, fontweight='bold')

    # Consumer Staples
    ax = axes[0]
    colors = [config.COLORS.get(c, '#333333') for c in available_components]
    markers = ['o', 's', '^'][:len(available_components)]

    for i, component in enumerate(available_components):
        ax.plot(yearly_cs['Year'], yearly_cs[component],
                marker=markers[i], linestyle='-', color=colors[i],
                linewidth=2, markersize=8, label=component)

    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('ESG Score', fontsize=12)
    ax.set_title('Consumer Staples Sector', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(yearly_cs['Year'].unique())
    ax.set_xticklabels(yearly_cs['Year'].unique(), rotation=45)

    # All Sectors
    ax = axes[1]
    for i, component in enumerate(available_components):
        ax.plot(yearly_all['Year'], yearly_all[component],
                marker=markers[i], linestyle='-', color=colors[i],
                linewidth=2, markersize=8, label=component)

    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('ESG Score', fontsize=12)
    ax.set_title('All Sectors', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(yearly_all['Year'].unique())
    ax.set_xticklabels(yearly_all['Year'].unique(), rotation=45)

    plt.tight_layout()

    # Save
    fig1_path = os.path.join(config.OUTPUT_DIR, 'fig1_esg_trends.png')
    plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {fig1_path}")

    # FIGURE 2: ESG Index (Normalized to first year=100)
    fig2, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig2.suptitle('ESG Index (First Year = 100)', fontsize=16, fontweight='bold')

    for sector_data, ax, title in [(yearly_cs, axes[0], 'Consumer Staples'),
                                   (yearly_all, axes[1], 'All Sectors')]:
        base_year = sector_data['Year'].min()
        base_values = sector_data[sector_data['Year'] == base_year][available_components].iloc[0]

        for i, component in enumerate(available_components):
            if base_values[component] != 0:
                index = (sector_data[component] / base_values[component]) * 100
                ax.plot(sector_data['Year'], index,
                        marker=markers[i], linestyle='-', color=colors[i],
                        linewidth=2, markersize=6, label=component)

        ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='Base Year')
        ax.set_xlabel('Year', fontsize=12)
        ax.set_ylabel('Index (Base Year=100)', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xticks(sector_data['Year'].unique())
        ax.set_xticklabels(sector_data['Year'].unique(), rotation=45)

    plt.tight_layout()

    # Save
    fig2_path = os.path.join(config.OUTPUT_DIR, 'fig2_esg_index.png')
    plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {fig2_path}")

    # FIGURE 3: Sector Comparison (each component separately)
    fig3, axes = plt.subplots(1, len(available_components), figsize=(15, 5))
    fig3.suptitle('Sector Comparison: Consumer Staples vs All Sectors', fontsize=16, fontweight='bold')

    if len(available_components) == 1:
        axes = [axes]

    for idx, component in enumerate(available_components):
        ax = axes[idx]
        ax.plot(yearly_cs['Year'], yearly_cs[component], 'o-',
                color=config.COLORS['cs'], linewidth=2, markersize=8,
                label='Consumer Staples')
        ax.plot(yearly_all['Year'], yearly_all[component], 's-',
                color=config.COLORS['all'], linewidth=2, markersize=8,
                label='All Sectors')
        ax.set_xlabel('Year', fontsize=12)
        ax.set_ylabel(f'{component} Score', fontsize=12)
        ax.set_title(f'{component} Component', fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xticks(yearly_cs['Year'].unique())
        ax.set_xticklabels(yearly_cs['Year'].unique(), rotation=45)

    plt.tight_layout()

    # Save
    fig3_path = os.path.join(config.OUTPUT_DIR, 'fig3_sector_comparison.png')
    plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {fig3_path}")

    return fig1, fig2, fig3


# ============================================================================
# FIGURE 4: R² COMPARISON ACROSS LAG MODELS
# ============================================================================
def create_r2_comparison_figure(analyzer_results, config):
    """Create R² comparison across moving average windows using actual results"""

    print("\n" + "="*60)
    print("CREATING R² COMPARISON VISUALIZATION")
    print("="*60)

    windows = [1, 2, 3, 4, 5]
    r2_cs = []
    r2_all = []
    window_labels = []

    for window in windows:
        key_cs = f"MA{window}_cs"
        key_all = f"MA{window}_all"

        # Get Consumer Staples R²
        if key_cs in analyzer_results:
            if 'r2' in analyzer_results[key_cs]:
                r2_cs.append(analyzer_results[key_cs]['r2'])
            elif 'results' in analyzer_results[key_cs]:
                r2_cs.append(analyzer_results[key_cs]['results'].rsquared)
            else:
                r2_cs.append(np.nan)
        else:
            r2_cs.append(np.nan)

        # Get All Sectors R²
        if key_all in analyzer_results:
            if 'r2' in analyzer_results[key_all]:
                r2_all.append(analyzer_results[key_all]['r2'])
            elif 'results' in analyzer_results[key_all]:
                r2_all.append(analyzer_results[key_all]['results'].rsquared)
            else:
                r2_all.append(np.nan)
        else:
            r2_all.append(np.nan)

        window_labels.append(f'{window}-Year MA' if window > 1 else 'Simple Lag')

    # Print extracted values
    print(f"R² values extracted:")
    for w, cs, all_ in zip(windows, r2_cs, r2_all):
        cs_str = f"{cs:.4f}" if not np.isnan(cs) else "N/A"
        all_str = f"{all_:.4f}" if not np.isnan(all_) else "N/A"
        print(f"  Window {w}: CS={cs_str}, All={all_str}")

    # Create the figure
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot lines
    if not all(np.isnan(r2_cs)):
        ax.plot(windows, r2_cs, 'o-', color=config.COLORS['cs'],
                linewidth=3, markersize=10, label='Consumer Staples')
        for i, (x, y) in enumerate(zip(windows, r2_cs)):
            if not np.isnan(y):
                ax.annotate(f'{y:.3f}', (x, y), textcoords="offset points",
                           xytext=(0,10), ha='center', fontsize=9, color=config.COLORS['cs'])

    if not all(np.isnan(r2_all)):
        ax.plot(windows, r2_all, 's-', color=config.COLORS['all'],
                linewidth=3, markersize=10, label='All Sectors')
        for i, (x, y) in enumerate(zip(windows, r2_all)):
            if not np.isnan(y):
                ax.annotate(f'{y:.3f}', (x, y), textcoords="offset points",
                           xytext=(0,-15), ha='center', fontsize=9, color=config.COLORS['all'])

    ax.set_xlabel('Moving Average Window', fontsize=14)
    ax.set_ylabel('R-squared', fontsize=14)
    ax.set_title('Model Fit Comparison: Consumer Staples vs All Sectors',
                fontsize=16, fontweight='bold')
    ax.set_xticks(windows)
    ax.set_xticklabels(window_labels)
    ax.legend(loc='best', fontsize=12)
    ax.grid(True, alpha=0.3)

    all_values = [v for v in r2_cs + r2_all if not np.isnan(v)]
    if all_values:
        y_min = max(0, min(all_values) - 0.05)
        y_max = min(1, max(all_values) + 0.05)
        ax.set_ylim([y_min, y_max])

    plt.tight_layout()

    # Save
    fig_path = os.path.join(config.OUTPUT_DIR, 'fig4_r2_comparison.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {fig_path}")

    return fig


# ============================================================================
# FIGURE 5: ALL TRANSFORMATIONS OF TOBIN'S Q
# ============================================================================
def create_all_transformations_figure(df_cs, config):
    """Create distribution plots for all Tobin's Q transformations"""

    print("\n" + "="*60)
    print("CREATING ALL TRANSFORMATIONS VISUALIZATIONS")
    print("="*60)

    if 'Tobin_Q' not in df_cs.columns:
        print("⚠ Tobin_Q not found in data")
        return None

    tobin_q = df_cs['Tobin_Q'].dropna()

    if len(tobin_q) == 0:
        print("⚠ No Tobin_Q data available")
        return None

    # Create all transformations
    transformations = {
        'Original': tobin_q,
        'Log': np.log(tobin_q + 0.001),
        'Square Root': np.sqrt(tobin_q.clip(lower=0)),
        'Inverse': 1 / tobin_q.replace(0, np.nan).dropna(),
    }

    # Add Box-Cox if possible
    try:
        if (tobin_q > 0).all():
            boxcox_data, lambda_param = stats.boxcox(tobin_q + 0.001)
            transformations[f'Box-Cox (λ={lambda_param:.2f})'] = boxcox_data
    except:
        pass

    # Create figure
    n_trans = len(transformations)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Tobin\'s Q: Distribution Across Transformations',
                fontsize=16, fontweight='bold')
    axes = axes.flatten()

    colors = ['#2E86AB', '#A23B72', '#F18F01', '#4C9A8A', '#9B59B6']

    for idx, (trans_name, data) in enumerate(transformations.items()):
        ax = axes[idx]
        ax.hist(data, bins=30, color=colors[idx % len(colors)],
                edgecolor='black', alpha=0.7)
        ax.axvline(data.mean(), color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {data.mean():.3f}')
        ax.axvline(data.median(), color='blue', linestyle=':', linewidth=2,
                   label=f'Median: {data.median():.3f}')
        ax.set_xlabel(trans_name, fontsize=12)
        ax.set_ylabel('Frequency', fontsize=12)
        ax.set_title(f'{trans_name} Distribution', fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=8)
        ax.grid(True, alpha=0.3)

        skewness = stats.skew(data)
        kurtosis = stats.kurtosis(data)
        ax.text(0.05, 0.95, f'Skew: {skewness:.3f}\nKurt: {kurtosis:.3f}',
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    for idx in range(len(transformations), len(axes)):
        axes[idx].set_visible(False)

    plt.tight_layout()

    # Save
    fig_path = os.path.join(config.OUTPUT_DIR, 'fig5_all_transformations.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {fig_path}")

    return fig


# ============================================================================
# FIGURE 6: Q-Q PLOTS FOR ALL TRANSFORMATIONS
# ============================================================================
def create_qq_plots_all(df_cs, config):
    """Create Q-Q plots for all transformations"""

    print("\n" + "="*60)
    print("CREATING Q-Q PLOTS FOR ALL TRANSFORMATIONS")
    print("="*60)

    if 'Tobin_Q' not in df_cs.columns:
        print("⚠ Tobin_Q not found in data")
        return None

    tobin_q = df_cs['Tobin_Q'].dropna()

    if len(tobin_q) == 0:
        print("⚠ No Tobin_Q data available")
        return None

    # Create all transformations
    transformations = {
        'Original': tobin_q,
        'Log': np.log(tobin_q + 0.001),
        'Square Root': np.sqrt(tobin_q.clip(lower=0)),
        'Inverse': 1 / tobin_q.replace(0, np.nan).dropna(),
    }

    try:
        if (tobin_q > 0).all():
            boxcox_data, lambda_param = stats.boxcox(tobin_q + 0.001)
            transformations['Box-Cox'] = boxcox_data
    except:
        pass

    # Create Q-Q plots
    n_trans = len(transformations)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Q-Q Plots: Normality Assessment Across Transformations',
                fontsize=16, fontweight='bold')
    axes = axes.flatten()

    for idx, (trans_name, data) in enumerate(transformations.items()):
        ax = axes[idx]
        stats.probplot(data, dist="norm", plot=ax)
        ax.set_title(f'{trans_name}', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)

        try:
            sample_data = data.sample(min(5000, len(data))) if len(data) > 5000 else data
            _, shapiro_p = stats.shapiro(sample_data)
            ax.text(0.05, 0.95, f'Shapiro-Wilk p={shapiro_p:.4f}',
                    transform=ax.transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        except:
            pass

    for idx in range(len(transformations), len(axes)):
        axes[idx].set_visible(False)

    plt.tight_layout()

    # Save
    fig_path = os.path.join(config.OUTPUT_DIR, 'fig6_qq_plots_all.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {fig_path}")

    return fig


# ============================================================================
# TABLE 1: VARIABLE DEFINITIONS
# ============================================================================
def create_variable_definition_table(config):
    """Create variable definition table"""

    print("\n" + "="*60)
    print("CREATING VARIABLE DEFINITION TABLE")
    print("="*60)

    definitions = [
        ['ROA', 'Net Income / Total Assets', 'Profitability measure', '+'],
        ['Tobin\'s Q', '(MVE + Total Liabilities) / Total Assets', 'Market valuation', 'N/A'],
        ['Size', 'Log(Total Assets)', 'Firm size control', '+/-'],
        ['Leverage', 'Total Liabilities / Total Assets', 'Capital structure', '-'],
        ['E', 'Environmental Score', 'ESG pillar', '+'],
        ['S', 'Social Score', 'ESG pillar', '+'],
        ['G', 'Governance Score', 'ESG pillar', '+'],
        ['E_lag1', 'E(t-1)', 'One-year lagged Environmental score', '+'],
        ['S_lag1', 'S(t-1)', 'One-year lagged Social score', '+'],
        ['G_lag1', 'G(t-1)', 'One-year lagged Governance score', '+'],
        ['E_ma3', 'Mean(E(t-1) to E(t-3))', '3-year moving average E', '+'],
        ['S_ma3', 'Mean(S(t-1) to S(t-3))', '3-year moving average S', '+'],
        ['G_ma3', 'Mean(G(t-1) to G(t-3))', '3-year moving average G', '+']
    ]

    df_table = pd.DataFrame(definitions,
                           columns=['Variable', 'Formula', 'Description', 'Expected Sign'])

    # Save as CSV
    csv_path = os.path.join(config.OUTPUT_DIR, 'table1_variable_definitions.csv')
    df_table.to_csv(csv_path, index=False)

    # Create LaTeX table
    latex_table = df_table.to_latex(index=False, caption='Variable Definitions',
                                    label='tab:variable_definitions', escape=False)

    tex_path = os.path.join(config.OUTPUT_DIR, 'table1_variable_definitions.tex')
    with open(tex_path, 'w') as f:
        f.write(latex_table)

    print(f"✓ Saved: {csv_path}")
    print(f"✓ Saved: {tex_path}")

    # Display
    print("\nVariable Definition Table:")
    print(df_table.to_string(index=False))

    return df_table


# ============================================================================
# TABLE 2: CORRELATION MATRIX
# ============================================================================
def create_correlation_matrix(df_cs, config):
    """Create correlation matrix with VIF diagnostics"""

    print("\n" + "="*60)
    print("CREATING CORRELATION MATRIX")
    print("="*60)

    # Select available variables
    base_vars = ['Tobin_Q', 'E', 'S', 'G', 'ROA', 'Size', 'Leverage']
    available_vars = [v for v in base_vars if v in df_cs.columns]

    if len(available_vars) < 2:
        print("⚠ Insufficient variables for correlation matrix")
        return None

    df_corr = df_cs[available_vars].dropna()

    if len(df_corr) < 10:
        print("⚠ Insufficient data for correlation matrix")
        return None

    # Calculate correlation matrix
    corr_matrix = df_corr.corr()

    # Create heatmap
    fig, ax = plt.subplots(figsize=(10, 8))

    # Mask for upper triangle
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    # Heatmap
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, square=True, linewidths=1,
                cbar_kws={"shrink": 0.8}, ax=ax)

    ax.set_title('Correlation Matrix - Consumer Staples Sector',
                fontsize=16, fontweight='bold')

    plt.tight_layout()

    # Save
    fig_path = os.path.join(config.OUTPUT_DIR, 'fig_correlation_matrix.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {fig_path}")

    # Calculate VIF for non-perfectly collinear variables
    try:
        # Exclude Tobin_Q from VIF calculation
        vif_vars = [v for v in available_vars if v != 'Tobin_Q']
        if len(vif_vars) >= 2:
            X = add_constant(df_corr[vif_vars])
            vif_data = pd.DataFrame()
            vif_data["Variable"] = X.columns
            vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

            # Add VIF to correlation table
            corr_with_vif = corr_matrix.copy()
            vif_series = vif_data.set_index('Variable')['VIF']
            # Only add VIF for columns that exist
            for col in corr_with_vif.columns:
                if col in vif_series.index:
                    corr_with_vif.loc['VIF', col] = vif_series[col]

            print("\nVariance Inflation Factors:")
            print(vif_data.to_string(index=False))
        else:
            corr_with_vif = corr_matrix
    except Exception as e:
        print(f"⚠ Could not calculate VIF: {e}")
        corr_with_vif = corr_matrix

    # Save correlation table
    csv_path = os.path.join(config.OUTPUT_DIR, 'table2_correlation_matrix.csv')
    corr_with_vif.to_csv(csv_path)

    # Display
    print("\nCorrelation Matrix:")
    print(corr_matrix.round(3))

    return corr_with_vif


# ============================================================================
# TABLE 3: SUMMARY STATISTICS BY SECTOR
# ============================================================================
def create_summary_statistics(df_cs, df_all, config):
    """Create summary statistics table comparing sectors"""

    print("\n" + "="*60)
    print("CREATING SUMMARY STATISTICS BY SECTOR")
    print("="*60)

    # Select available variables
    base_vars = ['Tobin_Q', 'E', 'S', 'G', 'ROA', 'Size', 'Leverage']
    available_vars = [v for v in base_vars if v in df_cs.columns]

    if not available_vars:
        print("⚠ No variables available for summary statistics")
        return None

    stats_list = []

    for sector_name, df in [('Consumer_Staples', df_cs), ('All_Sectors', df_all)]:
        sector_stats = df[available_vars].describe().T[['mean', 'std', 'min', 'max']]
        sector_stats.columns = [f'{sector_name}_{col}' for col in sector_stats.columns]
        stats_list.append(sector_stats)

    summary_stats = pd.concat(stats_list, axis=1)

    # Add observation counts
    obs_row = pd.Series({
        'Consumer_Staples_mean': len(df_cs),
        'Consumer_Staples_std': '',
        'Consumer_Staples_min': '',
        'Consumer_Staples_max': '',
        'All_Sectors_mean': len(df_all),
        'All_Sectors_std': '',
        'All_Sectors_min': '',
        'All_Sectors_max': ''
    }, name='Observations')

    # Ensure obs_row has the same index as summary_stats
    obs_row = obs_row.reindex(summary_stats.columns)
    summary_stats = pd.concat([summary_stats, obs_row.to_frame().T])

    # Format
    for col in summary_stats.columns:
        if '_mean' in col or '_std' in col:
            summary_stats[col] = summary_stats[col].apply(
                lambda x: f'{x:.4f}' if isinstance(x, (int, float)) else x
            )

    # Save
    csv_path = os.path.join(config.OUTPUT_DIR, 'table3_summary_statistics.csv')
    summary_stats.to_csv(csv_path)

    print("\nSummary Statistics by Sector:")
    print(summary_stats)
    print(f"✓ Saved: {csv_path}")

    return summary_stats


# ============================================================================
# MAIN INTEGRATION FUNCTION
# ============================================================================
def generate_annex_materials_from_analyzer(analyzer):
    """
    Generate all annex materials using existing analyzer object

    Parameters:
    -----------
    analyzer : MovingAverageAnalyzer
        Your analyzer object from the main analysis, containing:
        - analyzer.loader.df, analyzer.loader.df_cs, analyzer.loader.df_all
        - analyzer.results
    """

    print("\n" + "="*80)
    print("GENERATING ANNEX C VISUALIZATIONS AND TABLES")
    print("="*80)

    # Initialize config
    config = AnnexConfig()
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)

    # Get data from analyzer (NO RELOADING!)
    df_cs = analyzer.loader.df_cs
    df_all = analyzer.loader.df_all

    print(f"Using existing data - Consumer Staples: {len(df_cs)} obs, All Sectors: {len(df_all)} obs")

    # Create tables
    print("\n" + "#"*60)
    print("SECTION 1: CREATING TABLES")
    print("#"*60)

    var_table = create_variable_definition_table(config)
    corr_table = create_correlation_matrix(df_cs, config)
    summary_table = create_summary_statistics(df_cs, df_all, config)

    # Create figures
    print("\n" + "#"*60)
    print("SECTION 2: CREATING FIGURES")
    print("#"*60)

    # ESG Trends
    fig1, fig2, fig3 = create_esg_trend_figures(df_cs, df_all, config)

    # R² Comparison using actual results
    print("\nUsing actual analyzer results for R² comparison")
    fig4 = create_r2_comparison_figure(analyzer.results, config)

    # All Transformations of Tobin's Q
    fig5 = create_all_transformations_figure(df_cs, config)
    fig6 = create_qq_plots_all(df_cs, config)

    print("\n" + "="*80)
    print("✓ ALL ANNEX VISUALIZATIONS AND TABLES COMPLETE")
    print("="*80)
    print(f"\nOutput directory: {config.OUTPUT_DIR}")

    return {
        'tables': {
            'variable_definitions': var_table,
            'correlation': corr_table,
            'summary_statistics': summary_table
        },
        'figures': {
            'esg_trends': fig1,
            'esg_index': fig2,
            'sector_comparison': fig3,
            'r2_comparison': fig4,
            'all_transformations': fig5,
            'qq_plots': fig6
        },
        'output_dir': config.OUTPUT_DIR
    }


# ============================================================================
# IF RUN STANDALONE
# ============================================================================
if __name__ == "__main__":
    print("This module is designed to be imported and used with an existing analyzer object.")
    print("Please import and call generate_annex_materials_from_analyzer(analyzer)")


print("File created successfully!")

This module is designed to be imported and used with an existing analyzer object.
Please import and call generate_annex_materials_from_analyzer(analyzer)
File created successfully!
